# Pyomo Homework 2

## Course policies and submitting this assignment

Read the [Artificial Intelligence Policy](../../org/syllabus.md#artificial-intelligence-policy) and
[Collaboration Policy and Honor Code](../../org/syllabus.md#collaboration-policy-and-honor-code)
before starting. The assignment-specific directions below control where and when AI may be used.

Submit **two files to Canvas**:

1. **A scanned PDF of your handwritten work** for the pencil-and-paper problems. A clear phone
   photo assembled into a single PDF is fine.
2. **A copy of this notebook**, with your code cells run.

This assignment is graded on **completion**, not correctness. Published answers are provided so
you can check your own work — attempt each problem *before* you look.


```{warning}
**This is a draft assignment. It is still being updated for Fall 2026.**
```

In [ ]:
# This code cell installs packages on Colab

import sys

if "google.colab" in sys.modules:
    !wget "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/helper.py"
    import helper

    helper.easy_install()
else:
    sys.path.insert(0, "../")
    import helper
helper.set_plotting_style()

In [ ]:
## IMPORT LIBRARIES
import pyomo.environ as pyo
import pandas as pd

## How this assignment is organized

This homework has **five problems**. Problems 1, 2 and 5 are **code**. Problems 3 and 4 are
**pencil and paper** --- no solver, no code cell --- and are deliberately written in the format of
the in-person midterm, so that the exam is not the first time you see these questions.

| # | Problem | Format |
| --- | --- | --- |
| 1 | Pyomo fundamentals: the knapsack problem | code |
| 2 | Lot sizing | code |
| 3 | Big-$M$ and convex hull reformulations | pencil and paper |
| 4 | Generalized disjunctive programming (GDP) modeling | pencil and paper |
| 5 | Strip packing: big-$M$ versus convex hull in Pyomo | code |

Problems 3, 4 and 5 all come from **Logical Modeling and Generalized Disjunctive Programming**.
Work them in order: 3 asks you to reformulate a disjunction by hand, 4 asks you to *write* one from
an English specification, and 5 hands both jobs to Pyomo and asks what the two reformulations cost.

:::{important}
**AI category: AI permitted after independent work.** For each top-level problem, work for about
30 minutes without AI, solution pages, or help from another person, or stop early if you complete
the problem. You may consult lecture notes, textbooks, and nonsolution pages of the course website;
bias toward those course sources. Record your attempt before using AI. You may then use AI or
collaborate with classmates, but you must understand and verify the work. Complete the short AI use
report at the end of every problem.

In each report, state approximately how long the independent attempt took, how far you got, where you
became stuck, any AI or collaborative help used afterward, and how you verified it. If you used no AI,
say so. Use concise bullets; do not submit prompts or transcripts. Time estimates give the instructor
useful data for improving the assignment and are not a speed test.
:::


## Problem 1. Pyomo fundamentals: the knapsack problem

Parts **1-A** through **1-I** are all the same knapsack problem, developed from a first solve
through to enumerating near-optimal solutions with integer cuts. Part **1-J** is a short syntax
exercise on a different model.


Problems 1 and 2 are adapted from the Pyomo team's excellent PyomoFest workshop
(Bynum et al., 2021). Special thanks to them for creating these exercises.


### 1-A. Knapsack example

You want to fill a knapsack (a.k.a. bag). You can choose from a hammer, wrench, screwdriver, and towel. Each item has a different weight and value. You want to maximize the value (benefit) of the collection of items constrained by a total weight limit. Let's formulate this as an optimization problem.

**Sets**

$$\mathcal{A} = \{\text{hammer},~\text{wrench},~\text{screwdriver},~\text{towel} \}$$  

**Parameters (Data)**

Let $b_i$ and $w_i$ represent the benefit and weight of item $i$, respectively.

| Item ($i$)  | Benefit ($b_i$) | Weight ($w_i$) |
| ----------- | ----------- | ----------- |
| hammer      | 8      | 5|
| wrench   | 3        | 7 |
| screwdriver  | 6 | 4        |
| towel   | 11  | 3 |

Let $W_{max} = 14$ be the maximum weight.

**Variables**

Let $x_i \in \{0,1\}$ (binary) represent whether or not we include item $i$ in the knapsack. For now, we will consider only being able to choose either none or one of each item.

**Objective and Constraints**

$$
\begin{equation} 
\begin{split}
\max_{x} \quad & \sum_{i\in{\mathcal{A}}}b_i x_i \\
\text{s.t.} \quad & \sum_{i\in{\mathcal{A}}}w_ix_i \leq W_{max} \\
& x_i \in \{0,1\}, \quad \forall i \in \mathcal{A}
\end{split}
\end{equation}
$$


**Pyomo**

Solve the knapsack problem given below using [HiGHS](https://highs.dev/) and answer the following questions:

1. Which items are acquired in the optimal solution?

2. Why does this solution make sense? (Write ~2 sentences.)

We use [HiGHS](https://highs.dev/), a modern open-source solver for linear and mixed-integer linear programs, which we call from Pyomo as `pyo.SolverFactory('appsi_highs')`. Earlier versions of this assignment used GLPK. HiGHS is faster, is actively developed, and installs anywhere with `pip install highspy`.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

# Add your solution here

model.display()

**Question Answers**

1. *Fill in here*

2. *Fill in here*

### 1-B. Knapsack example with improved printing

Complete the missing lines in the code below to produce formatted output: print the total weight, the value of the items selected (the objective), and the items acquired in the optimal solution. Note, the Pyomo value function should be used to get the floating point value of Pyomo modeling components (e.g., `print(value(model.x[i])`).

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

opt = pyo.SolverFactory("appsi_highs")
opt_success = opt.solve(model)
assert pyo.check_optimal_termination(opt_success), (
    f"Solve failed: status={opt_success.solver.status}, "
    f"termination={opt_success.solver.termination_condition}"
)

total_weight = sum(w[i] * pyo.value(model.x[i]) for i in A)
# Add your solution here

print("%12s %12s" % ("Item", "Selected"))
print("=========================")
for i in A:
    acquired = "No"
    # Add your solution here
print("-------------------------")

### 1-C. Changing data

Using your code from **1-B**, if we were to increase the value of the wrench, at what point would it become selected as part of the optimal solution?

In [ ]:
# Add your solution here

**Question Answer**

*Fill in here*

### 1-D. Loading data from Excel

In the code above, the data is hardcoded at the top of the file. Instead of hardcoding the data, use Python to load the data from a different source. You may use Pandas to load data from 'knapsack_data.xlsx' into a dataframe. You will then need to write code to obtain a dictionary from the dataframe.

In [ ]:
df_items = pd.read_excel(
    "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/data/knapsack_data.xlsx", sheet_name="data", header=0, index_col=0
)
W_max = 14

A = df_items.index.tolist()
# Add your solution here

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

opt = pyo.SolverFactory("appsi_highs")
opt_success = opt.solve(model, tee=True)
assert pyo.check_optimal_termination(opt_success), (
    f"Solve failed: status={opt_success.solver.status}, "
    f"termination={opt_success.solver.termination_condition}"
)

total_weight = sum(w[i] * pyo.value(model.x[i]) for i in A)
print("Total Weight:", total_weight)
print("Total Benefit:", pyo.value(model.obj))

print("%12s %12s" % ("Item", "Selected"))
print("=========================")
for i in A:
    acquired = "No"
    if pyo.value(model.x[i]) >= 0.5:
        acquired = "Yes"
    print("%12s %12s" % (i, acquired))
print("-------------------------")

### 1-E. NLP vs. MIP

Solve the knapsack problem with IPOPT instead of HiGHS. Print the solution values for model.x. What happened? Why?

*Hint*: Switch `appsi_highs` to `ipopt` in the call to `SolverFactory`.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(b[i] * model.x[i] for i in A), sense=pyo.maximize)

model.weight_con = pyo.Constraint(expr=sum(w[i] * model.x[i] for i in A) <= W_max)

# Add your solution here
opt_success = opt.solve(model, tee=True)
assert pyo.check_optimal_termination(opt_success), (
    f"Solve failed: status={opt_success.solver.status}, "
    f"termination={opt_success.solver.termination_condition}"
)

model.pprint()

**Question Answers**

*Fill in here*

### 1-F. Knapsack problem with rules

Rules are important for defining indexed constraints, however, they can also be used for single (i.e. scalar) constraints. Reimplement the knapsack model from **1-A** using rules for the objective and the constraints.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)


# Add your solution here

### 1-G. Integer formulation of the knapsack problem

Consider again the knapsack problem. Assume now that we can acquire multiple items of the same type. In this new formulation, $x_i$ is now an integer variable instead of a binary variable. One way to formulate this problem is as follows:

$$
\begin{equation} 
\begin{split}
\max_{x} \quad & \sum_{i\in{\mathcal{A}}}b_i x_i \\
\text{s.t.} \quad & \sum_{i\in{\mathcal{A}}}w_i x_i \leq W_{max} \\
 & x_i=\sum_{j=0}^Njq_{i,j}, \quad \forall i \in \mathcal{A} \\
 & 0 \leq x_i \leq N, \quad \forall i \in \mathcal{A} \\
 & q_{i,j} \in \{0,1\}, \quad \forall i \in \mathcal{A}, j \in \{0,...,N\}
\end{split}
\end{equation}
$$

One could optionally add the following constraint to select only one $q_{i,j}$ for each $i$, although it is not strictly necessary to yield an integer solution.
$$
\begin{equation}
\sum_{j=0}^N q_{i,j} = 1, \quad \forall i \in \mathcal{A}
\end{equation}
$$

Starting with your code from **1-F**, implement this new formulation and solve. Is the solution surprising?

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14
N = range(6)  # create a list from 0-5

model = pyo.ConcreteModel()

model.x = pyo.Var(A)
model.q = pyo.Var(A, N, domain=pyo.Binary)


def obj_rule(m):
    return sum(b[i] * m.x[i] for i in A)


model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)


def weight_con_rule(m):
    return sum(w[i] * m.x[i] for i in A) <= W_max


model.weight_con = pyo.Constraint(rule=weight_con_rule)


# Add your solution here

**Question Answer**

*Fill in here*

### 1-H. Changing parameter values with a mutable `Param`

A parameter can be specified to be mutable. This tells Pyomo that the value of the parameter may change in the future, and allows the user to change the parameter value and resolve the problem without the need to rebuild the entire model each time. We will use this functionality to find a better solution to the knapsack problem. We would like to find when the wrench becomes valuable enough to be a part of the optimal solution. Create a Pyomo Parameter for the value of the items, make it mutable, and then write a loop that prints the solution for different wrench values.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)
# Add your solution here


def obj_rule(m):
    return sum(m.item_benefit[i] * m.x[i] for i in A)


model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)


def weight_rule(m):
    return sum(w[i] * m.x[i] for i in A) <= W_max


model.weight = pyo.Constraint(rule=weight_rule)

# You may instead use 'cbc' as the solver
opt = pyo.SolverFactory("appsi_highs")

for wrench_benefit in range(1, 11):
    model.item_benefit["wrench"] = wrench_benefit
    result_obj = opt.solve(model)
    assert pyo.check_optimal_termination(result_obj), (
        f"Solve failed: status={result_obj.solver.status}, "
        f"termination={result_obj.solver.termination_condition}"
    )

    # Add your solution here

### 1-I. Integer cuts

Often, it can be important to find not only the "best" solution, but a number of solutions that are equally optimal, or close to optimal. For discrete optimization problems, this can be done using something known as an integer cut. Consider again the knapsack problem where the choice of which items to select is a discrete variable $x_i \forall i \in A$. Let $x_i^*$ be a particular set of $x$ values we want to remove from the feasible solution space. We define an integer cut using two sets. The first set $S_0$ contains the indices for those variables whose current solution is 0, and the second set $S_1$ consists of indices for those variables whose current solution is 1. Given these two sets, an integer cut constraint that would prevent such a solution from appearing again is defined by,

$\sum_{i \in S_0}x[i] + \sum_{i \in S_1}(1-x_i) \geq 1$

Write a loop that solves the problem 5 times, adding an integer cut to remove the previous solution, and printing the value of the objective function and the solution at each iteration of the loop.

In [ ]:
A = ["hammer", "wrench", "screwdriver", "towel"]
b = {"hammer": 8, "wrench": 3, "screwdriver": 6, "towel": 11}
w = {"hammer": 5, "wrench": 7, "screwdriver": 4, "towel": 3}
W_max = 14

model = pyo.ConcreteModel()
model.x = pyo.Var(A, domain=pyo.Binary)


def obj_rule(m):
    return sum(b[i] * m.x[i] for i in A)


model.obj = pyo.Objective(rule=obj_rule, sense=pyo.maximize)


def weight_con_rule(m):
    return sum(w[i] * m.x[i] for i in A) <= W_max


model.weight_con = pyo.Constraint(rule=weight_con_rule)

# You may instead use 'cbc' as the solver
opt = pyo.SolverFactory("appsi_highs")

# create the ConstraintList to hold the integer cuts
model.int_cuts = pyo.ConstraintList()

# Add your solution here

### 1-J. Decorator notation

Alternative notation for declaring and defining Pyomo components using decorators exists. Starting with the warehouse location problem code below, change the model to use the decorator notation.

This is the last part of Problem 1 and the only one that is not the knapsack problem: the point is the *syntax*, not the model. **Problem 5 builds its model entirely with decorators**, including `@model.Disjunction`, so this is the notation you will need there.


In [ ]:
# warehouse_location.py: Warehouse location determination problem
model = pyo.ConcreteModel(name="(WL)")

W = ["Harlingen", "Memphis", "Ashland"]
C = ["NYC", "LA", "Chicago", "Houston"]
d = {
    ("Harlingen", "NYC"): 1956,
    ("Harlingen", "LA"): 1606,
    ("Harlingen", "Chicago"): 1410,
    ("Harlingen", "Houston"): 330,
    ("Memphis", "NYC"): 1096,
    ("Memphis", "LA"): 1792,
    ("Memphis", "Chicago"): 531,
    ("Memphis", "Houston"): 567,
    ("Ashland", "NYC"): 485,
    ("Ashland", "LA"): 2322,
    ("Ashland", "Chicago"): 324,
    ("Ashland", "Houston"): 1236,
}
P = 2

model.x = pyo.Var(W, C, bounds=(0, 1))
model.y = pyo.Var(W, domain=pyo.Binary)


@model.Objective()
def obj(m):
    return sum(d[w, c] * m.x[w, c] for w in W for c in C)


@model.Constraint(C)
def one_per_cust(m, c):
    return sum(m.x[w, c] for w in W) == 1


# Add your solution here
def warehouse_active(m, w, c):
    return m.x[w, c] <= m.y[w]


# Note: This is only split across cells because of a bug in nbpages (notebook/website software).
# There is no other reason to split your code across cells.

In [ ]:
# Add your solution here
def num_warehouses(m):
    return sum(m.y[w] for w in W) <= P


results = pyo.SolverFactory("appsi_highs").solve(model)
assert pyo.check_optimal_termination(results), (
    f"Solve failed: status={results.solver.status}, "
    f"termination={results.solver.termination_condition}"
)

model.y.pprint()
model.x.pprint()

### Problem 1 AI use report

Use the concise report format stated at the top of the assignment.

## Problem 2. Lot sizing (Bynum et al., 2021)

We will now write a complete model from scratch using a well-known multi-period optimization problem for optimal lot-sizing adapted from Haugen et al. (2001) shown below.

$\min \sum_{t \in T}c_ty_t+h_t^+I_t^+ +h_t^-I_t^-$

s.t. $I_t=I_{t-1}+X_t-d_t, \forall t \in T$

$I_t=I_t^+-I_t^-, \forall t \in T$

$X_t \leq Py_t, \forall t \in T$

$X_t, I_t^+, I_t^- \geq 0, \forall t \in T$

$y_t \in \{0,1\}, \forall t \in T$

Our goal is to find the optimal production $X_t$ given known demands $d_t$, fixed cost $c_t$ associated with active production in a particular time period, an inventory holding cost $h_t^+$ and a shortage cost $h_t^-$ (cost of keeping a backlog) of orders. The variable $y_t$ (binary) determines if we produce in time $t$ or not, and $I_t^+$ represents inventory that we are storing across time period $t$, while $I_t^-$ represents the magnitude of the backlog. Note that $X_t \leq Py_t$ is a constraint that only allows production in time period $t$ if the indicator variable $y_t$=1.

Write a Pyomo model for this problem and solve it using HiGHS, i.e., `pyo.SolverFactory('appsi_highs')`, (or cbc) using the data provided below.

|Parameter|Description|Value|
|---|---|---|
|$c$|fixed cost of production|4.6|
|$I_0^+$|initial value of positive inventory|5.0|
|$I_0^-$|initial value of backlogged orders|0.0|
|$h^+$|cost (per unit) of holding inventory|0.7|
|$h^-$|shortage cost (per unit)|1.2|
|$P$|maximum production amount (big-M value)|5|
|$d$|demand|[5, 7, 6.2, 3.1, 1.7]|

**Reference**: Bynum, M. L., Hackebeil, G. A., Hart, W. E., Laird, C. D., Nicholson, B. L., Siirola, J. D., Watson, J.-P., and Woodruff, D. L. *Pyomo — Optimization Modeling in Python*, Third Edition. Springer Optimization and Its Applications, Vol. 67, 2021. (§8.6, p. 117)

In [ ]:
model = pyo.ConcreteModel()
model.T = pyo.RangeSet(5)  # time periods

i0 = 5.0  # initial inventory
c = 4.6  # setup cost
h_pos = 0.7  # inventory holding cost
h_neg = 1.2  # shortage cost
P = 5.0  # maximum production amount

# demand during period t
d = {1: 5.0, 2: 7.0, 3: 6.2, 4: 3.1, 5: 1.7}

# Add your solution here

# solve the problem
# You may instead use 'cbc' as the solver
solver = pyo.SolverFactory("appsi_highs")
results = solver.solve(model)
assert pyo.check_optimal_termination(results), (
    f"Solve failed: status={results.solver.status}, "
    f"termination={results.solver.termination_condition}"
)

# print the results
for t in model.T:
    print("Period: {0}, Prod. Amount: {1}".format(t, pyo.value(model.x[t])))

### Problem 2 AI use report

Use the concise report format stated at the top of the assignment.

## Problem 3. Big-$M$ and convex hull reformulations

*Pencil and paper. No solver, no code cell. This problem is written in the format of the in-person
midterm.*

**Background.** A process must deliver a flow rate $F$ [kmol/h]. **Exactly one** of three pump
types is installed, and each type has its own operating window and its own installed cost
$\gamma_i$ [thousand USD]:

| Pump $i$ | Operating window [kmol/h] | Installed cost $\gamma_i$ [thousand USD] |
| --- | --- | --- |
| 1 | $5 \leq F \leq 20$ | 12 |
| 2 | $25 \leq F \leq 45$ | 20 |
| 3 | $60 \leq F \leq 90$ | 35 |

The windows are hard physical limits, not preferences: outside its window a pump cannot run at all.
Independently of which pump is installed, the flow rate is bounded by $0 \leq F \leq 90$ kmol/h.

**Instruction.** Use the notation of the *Logical Modeling and Generalized Disjunctive Programming*
lecture throughout. Boolean indicators are capital $Y_i$; the binary variables that represent them
are lower-case $y_i$; and every inequality is written in the standard form $g(x) \leq 0$, so
$F \geq 5$ is written $-F \leq -5$.


### 3-A. Write the pump selection as a disjunction

Write the selection in generalized disjunctive programming standard form. Give the index set, the
Boolean indicators, the constraints inside each disjunct, the cost assignment, and the logical
constraint $\Omega(Y)$.

Next to each set, indicator, and equation, write a few-word description.

**Sets.**

**Parameters (Data).** *Hint*: state the units of each one.

**Variables.**

**The disjunction.**

**$\Omega(Y)$.**

Then answer in one sentence: **why can this not be written as a single pair of bounds on $F$?**


### 3-B. Big-$M$ reformulation

1. Write the big-$M$ reformulation of your disjunction. Introduce a binary $y_i$ for each Boolean
   $Y_i$ and write out **all six** relaxed inequalities explicitly, plus the constraint that
   selects one pump.
2. Determine the **smallest valid** $M_i$ for each of the three terms, using one $M_i$ per term.
   Show the reasoning, not just the number. *Hint*: $M_i$ must dominate the largest violation the
   term's constraints can attain anywhere in $0 \leq F \leq 90$.
3. In one or two sentences each: what goes wrong if $M_i$ is chosen **too large**, and what goes
   wrong if it is chosen **too small**? The two failures are not the same kind of failure.
4. You could instead use a **separate $M$ for every inequality** rather than one per term. Would
   that be tighter, looser, or the same? Give the six values.


### 3-C. Convex hull reformulation

1. Write the disaggregated convex hull reformulation. Disaggregate $F$ into $F_1$, $F_2$, $F_3$ ---
   one copy per pump --- and write out all the constraints explicitly.
2. The general form carries an **optional** bound $0 \leq z_i \leq U y_i$. Is it needed here?
   Justify your answer by setting $y_i = 0$ and reading off what the term's own constraints force.
3. Note that the cost $\gamma_i$ needs no big-$M$ and no disaggregation. Write $c$, the installed
   cost, as a single linear equation in the $y_i$, and explain in one sentence why that equation is
   exact rather than a relaxation.


### 3-D. Compare the two relaxations

This is the part that explains why anyone pays for the extra variables in 3-C.

1. **Relax the integrality** ($y_i \in \{0,1\}$ becomes $0 \leq y_i \leq 1$) and fix
   $y = (\tfrac{1}{2}, \tfrac{1}{2}, 0)$. Using your smallest valid per-term $M_i$ from 3-B,
   compute the interval of $F$ that the **big-$M$** relaxation admits, and the interval that the
   **convex hull** relaxation admits. Show the arithmetic.
2. Which interval is contained in the other? Is the containment strict?
3. Now let $y$ range freely over $0 \leq y_i \leq 1$ with $\sum_i y_i = 1$. What is the **full
   projection** onto $F$ of each relaxation --- that is, the set of $F$ values that survive for
   *some* fractional $y$? Compare each answer to the convex hull of
   $[5, 20] \cup [25, 45] \cup [60, 90]$, which is what the convex hull reformulation is
   *supposed* to give you.
4. One of the two answers in part 3 should alarm you. Say in two sentences what it means for
   branch and bound.


### 3-E. Problem size

Let $|D|$ be the number of disjunction terms, let $x \in \mathbb{R}^n$ be the continuous variables
appearing inside the disjuncts, and let term $i$ carry $m_i$ inequality constraints, with
$m = \sum_{i \in D} m_i$. Using these symbols, determine the size of each reformulation. After each
header below, give a number with a brief justification, **for big-$M$ and for the convex hull
separately**.

**Number of continuous variables:**

**Number of integer/discrete variables:**

**Number of equality constraints:**

**Number of inequality constraints:**

Then fill in the numbers for the pump problem specifically ($n = 1$, $|D| = 3$, $m = 6$).


### 3-F. Short answer

Two or three sentences each, not a short essay.

1. You must hand this model to two different audiences: a **colleague** who has to understand what
   the plant does, and a **solver**. Which of the three forms (the disjunction, big-$M$, the convex
   hull) goes to each, and why?
2. Under what circumstances would you deliberately choose big-$M$ even though 3-D shows its
   relaxation is worse?


### Problem 3 AI use report

Use the concise report format stated at the top of the assignment.

## Problem 4. Generalized disjunctive programming (GDP) modeling

*Pencil and paper. No solver, no code cell. This problem is written in the format of the in-person
midterm.*

**Background.** You are screening a superstructure for a new gas-processing plant. The design has
two reactor options and three separator options:

| Symbol | Unit |
| --- | --- |
| $R_1$ | low-conversion reactor (cheap) |
| $R_2$ | high-conversion reactor (expensive) |
| $A$ | absorber |
| $M$ | membrane |
| $CS$ | cryogenic separation |

Write $P_u$ for the proposition *"unit $u$ is installed"*, and let $y_u$ be the corresponding binary
variable.

The process engineers hand you six design rules, in English:

1. **Exactly one** reactor is installed.
2. **At least one** separator is installed.
3. Cryogenic separation is only worth its capital cost when the high-conversion reactor is
   installed.
4. The absorber and the membrane cannot **both** be installed --- they compete for the same plot
   space.
5. If the high-conversion reactor is installed, then cryogenic separation or the membrane (or both)
   must be installed --- that reactor's effluent carries light ends the absorber alone cannot
   remove.
6. If cryogenic separation is installed **and** the absorber is not, then the membrane must be
   installed.

The reactor choice also fixes the achievable conversion window and the reactor capital cost $c_R$
[million USD]:

| Reactor | Conversion window | $c_R$ [million USD] |
| --- | --- | --- |
| $R_1$ | $0.30 \leq X \leq 0.60$ | 40 |
| $R_2$ | $0.80 \leq X \leq 0.95$ | 90 |

Conversion is physically bounded by $0 \leq X \leq 1$.

**Instruction.** Same notation rules as Problem 3: capital $Y$ for Booleans, lower-case $y$ for
binaries, inequalities in the standard form $g(x) \leq 0$.


### 4-A. Translate the design into a mathematical model using set notation similar to our in-class examples

Next to each set, parameter, and variable, write a few-word description.

**Sets.**

**Parameters (Data).** *Hint*: state the units of each one.

**Variables.** Say for each one whether it is continuous, binary, or Boolean --- and be explicit
about the relationship between a Boolean $Y_u$ and its binary $y_u$.


### 4-B. Write each design rule as a logic proposition

Write rules 1 through 6 in the symbols $P_{R_1}, P_{R_2}, P_A, P_M, P_{CS}$ using
$\neg$, $\wedge$, $\vee$, $\veebar$ and $\Rightarrow$. Do **not** convert to constraints yet.

Watch the parentheses. Rules 5 and 6 are the two where a careless reading gives the wrong
proposition.


### 4-C. Convert the propositions to linear constraints

Convert each of rules 1 through 6 into linear constraints on the binaries $y_u$.

For rules 1, 2, 3 and 4 you may quote the translation table from lecture --- give the row you used.

For rules **5 and 6**, show the full three-step derivation:

&nbsp;&nbsp;&nbsp;&nbsp;① replace the implication &nbsp; ② apply De Morgan &nbsp; ③ distribute
$\vee$ over $\wedge$ to reach conjunctive normal form,

then substitute $y_u$ for $P_u$ and $1 - y_u$ for $\neg P_u$, and write one constraint per clause.

Finally, **sanity-check your rule 6 constraint** by substituting a case where the antecedent fires
and a case where it does not.


### 4-D. Write the reactor choice as a disjunction

The conversion window and the reactor cost are *not* logic on binaries --- they switch a block of
constraints on a **continuous** variable in or out. Write the reactor choice in GDP standard form:
the disjunction over the reactor options, with the conversion bounds and the cost assignment inside
each disjunct, and the appropriate $\Omega(Y)$.

Then answer: **which of your six rules from 4-B is now redundant**, because $\Omega(Y)$ already
says it?


### 4-E. Problem size

Let $N_r$ be the number of reactor options and $N_s$ the number of separator options. Using these
symbols, determine the size of the model **after** the disjunction has been reformulated with
big-$M$. After each header below, give a number with a brief justification.

**Number of continuous variables:**

**Number of integer/discrete variables:**

**Number of equality constraints:**

**Number of inequality constraints:**

Then answer the interesting part: **which of these counts actually grow with $N_r$ and $N_s$, and
which do not?** Look carefully at the six logic constraints before you answer.


### 4-F. Classify the problem

Is the reformulated model an LP, QP, NLP, MILP or MINLP? Justify your classification by referring
to the objective, the constraints and the variable domains.

Then: **is the problem convex?** Be careful --- answer separately for the model itself and for its
relaxation, and connect your answer to what you found in Problem 3-D.


### Problem 4 AI use report

Use the concise report format stated at the top of the assignment.

## Problem 5. Strip packing: big-$M$ versus convex hull in Pyomo

Problems 3 and 4 asked you to reformulate a disjunction by hand, on a problem small enough to see
all of. This problem hands both reformulations to Pyomo on a problem that is *not* small, and asks
what each one costs.

**Background.** Eight rectangles must be packed, without rotation and without overlap, into a strip
of fixed width $W = 10$. Rectangle $i \in N$ has length $L_i$ (along the strip) and height $H_i$
(across it), and is placed by the coordinates $(x_i, y_i)$ of its lower-left corner. The objective
is to minimize the length of strip used, $lt$:

$$\min_{x, y, lt} \; lt \qquad \text{s.t.} \qquad lt \geq x_i + L_i \quad \forall i \in N$$

Non-overlap is a disjunction for every **pair** of rectangles $(i,j)$ with $i < j$, with **four**
disjuncts --- $i$ is left of $j$, $i$ is right of $j$, $i$ is below $j$, or $i$ is above $j$:

$$
\begin{bmatrix} Y_{ij}^{1} \\ x_{i} + L_{i} \leq x_{j} \end{bmatrix}
\vee
\begin{bmatrix} Y_{ij}^{2} \\ x_{j} + L_{j} \leq x_{i} \end{bmatrix}
\vee
\begin{bmatrix} Y_{ij}^{3} \\ y_{i} + H_{i} \leq y_{j} \end{bmatrix}
\vee
\begin{bmatrix} Y_{ij}^{4} \\ y_{j} + H_{j} \leq y_{i} \end{bmatrix}
$$

Nothing about the method changes from Problem 3 --- there are just $\binom{8}{2} = 28$ disjunctions
instead of one, and four terms instead of three.

**Reference.** Strip packing instance from the MINLP library,
[https://www.minlp.org/library/problem/index.php?i=121&lib=GDP](https://www.minlp.org/library/problem/index.php?i=121&lib=GDP)
(Vecchietti and Grossmann). The class notebook
[](../2/Modeling_Disjunctions_Strip_Packing.ipynb) works this model in full --- attempt **5-A**
yourself before opening it.

:::{note}
**A lower bound you can compute by hand.** The rectangles have total area
$\sum_i L_i H_i = 109$, and the strip is $W = 10$ wide, so **no packing shorter than
$109/10 = 10.9$ exists.** Keep that number in mind; you will need it in 5-D.
:::


### 5-A. Build the GDP model

Complete `create_model()` below by adding the non-overlap disjunctions. Use the `@model.Disjunction`
decorator (this is the notation you practised in **1-J**) indexed over `model.overlap_pairs`, and
return the four disjuncts as a Python list of expressions.

**Do not** write big-$M$ or convex hull constraints by hand. The whole point of writing the
disjunction is that the transformation is somebody else's job.


In [ ]:
from pyomo.environ import (
    check_optimal_termination,
    ConcreteModel,
    Constraint,
    NonNegativeReals,
    Objective,
    Param,
    Set,
    SolverFactory,
    TransformationFactory,
    Var,
    value,
)


def create_model():
    """Build the strip packing problem as a generalized disjunctive program.

    Returns:
        model: Pyomo model, with the no-overlap disjunctions attached but NOT
            yet reformulated into a MILP.
    """
    model = ConcreteModel(name="Rectangles strip packing")

    ## Sets
    model.rectangles = Set(ordered=True, initialize=[0, 1, 2, 3, 4, 5, 6, 7])

    ## Parameters
    # Extent of each rectangle across the width of the strip (the y direction)
    model.rect_width = Param(
        model.rectangles, initialize={0: 3, 1: 3, 2: 2, 3: 2, 4: 3, 5: 5, 6: 7, 7: 7}
    )

    # Extent of each rectangle along the strip (the x direction)
    model.rect_length = Param(
        model.rectangles, initialize={0: 4, 1: 3, 2: 2, 3: 2, 4: 3, 5: 3, 6: 4, 7: 4}
    )

    model.strip_width = Param(initialize=10, doc="Available width of the strip")

    # Upper bound on length: every rectangle laid end to end
    model.max_length = Param(
        initialize=sum(model.rect_length[i] for i in model.rectangles)
    )

    ## Variables
    model.x = Var(
        model.rectangles,
        bounds=(0, model.max_length),
        doc="Rectangle corner position along the strip",
    )

    def w_bounds(b, i):
        return (0, b.strip_width - b.rect_width[i])

    model.y = Var(
        model.rectangles, bounds=w_bounds, doc="Rectangle corner position across the strip"
    )

    model.strip_length = Var(domain=NonNegativeReals, doc="Length of strip required")

    # The 28 unordered pairs of rectangles
    model.overlap_pairs = Set(
        initialize=model.rectangles * model.rectangles,
        dimen=2,
        filter=lambda b, i, j: i < j,
        doc="Set of possible rectangle conflicts",
    )

    ## Constraints
    @model.Constraint(model.rectangles)
    def strip_ends_after_last_rec(b, i):
        return b.strip_length >= b.x[i] + b.rect_length[i]

    ## Objective
    model.total_length = Objective(expr=model.strip_length, doc="Minimize length")

    ## Add the no-overlap disjunctions here!

    # Add your solution here

    return model


print(f"{len(create_model().overlap_pairs)} pairs of rectangles, one disjunction each")


### 5-B. Measure the size of a model

Before solving anything, write a function that reports the **size** of a Pyomo model: how many
continuous variables, how many binary variables, and how many active constraints it has.

This is the same count you did by hand in **3-E**, now done by the software. Use
`model.component_data_objects(Var, active=True)` and the same for `Constraint`, and test each
variable with `.is_binary()` and `.is_continuous()`.


In [ ]:
def model_size(model):
    """Count the continuous variables, binary variables, and active constraints.

    Argument:
        model: a Pyomo model, AFTER a GDP transformation has been applied

    Returns:
        dict with keys "continuous", "binary", "constraints"
    """
    # Add your solution here
    return {"continuous": n_cont, "binary": n_bin, "constraints": n_con}


### 5-C. Solve with the big-$M$ reformulation

Apply `TransformationFactory("gdp.bigm")` to a fresh model, solve it with HiGHS, and print the
optimal strip length and the placement of every rectangle.

*Hint*: the solver returns coordinates a few floating-point units off a whole number
(`6.999999999999998`, `-0.0`). Print with a format like `%.4g` --- that is numerical noise, not
geometry.


In [ ]:
def solve_variant(transformation, tee=False):
    """Build, transform, and solve the strip packing model.

    Argument:
        transformation: "gdp.bigm" or "gdp.hull"

    Returns:
        (model, size) -- the solved model and its size dict
    """
    model = create_model()
    # Add your solution here
    size = model_size(model)
    results = SolverFactory("appsi_highs").solve(model, tee=tee)
    assert check_optimal_termination(results), (
        f"Solve failed: status={results.solver.status}, "
        f"termination={results.solver.termination_condition}"
    )
    return model, size


bigm_model, bigm_size = solve_variant("gdp.bigm")

print("big-M reformulation")
print(f"  optimal strip length lt = {value(bigm_model.total_length):.4g}")
for i in bigm_model.rectangles:
    # +0.0 turns the solver's -0.0 into 0.0; .4g hides floating-point noise
    # such as 6.999999999999998, which is arithmetic, not geometry.
    xi = value(bigm_model.x[i]) + 0.0
    yi = value(bigm_model.y[i]) + 0.0
    print(f"  rectangle {i} at ({xi:.4g}, {yi:.4g})")
print(" ", bigm_size)


### 5-D. Solve with the convex hull reformulation, and compare

Do the same with `gdp.hull`. Then build the comparison.

🔴 **Do not compare the two by wall-clock time.** A solve time depends on the machine, the solver
version, and what else the computer is doing, so a timing you print here is not a number your
classmate --- or your grader --- can reproduce. Compare the two on things that are properties of
the *model*:

1. **Size**: continuous variables, binary variables, constraints, from `model_size`.
2. **Tightness of the relaxation**: relax every binary to $0 \leq y \leq 1$ with
   `TransformationFactory("core.relax_integer_vars")` and solve the resulting LP. Its objective is
   the bound branch and bound starts from at the root node.

Report a table with both, plus the MILP optimum and the $109/10 = 10.9$ area bound from the note
above. Then answer, in a few sentences each:

1. Do the two reformulations reach the **same optimal $lt$**? Do they reach it by the **same
   placement**? (5-E will show you.)
2. Which relaxation is tighter, and by how much? Compare each root bound to the true optimum and to
   the area bound.
3. The hull model is several times larger. Which *kind* of variable did it add --- and why does
   that make the trade worth taking? Connect this to your answer in **3-E**.


In [ ]:
def root_relaxation_bound(transformation):
    """Objective of the LP relaxation: the bound branch and bound starts from.

    Fully deterministic and machine-independent, unlike a solve time.
    """
    model = create_model()
    TransformationFactory(transformation).apply_to(model)
    # Add your solution here
    results = SolverFactory("appsi_highs").solve(model)
    assert check_optimal_termination(results)
    return value(model.total_length)


hull_model, hull_size = solve_variant("gdp.hull")

print("convex hull reformulation")
print(f"  optimal strip length lt = {value(hull_model.total_length):.4g}")
for i in hull_model.rectangles:
    # +0.0 turns the solver's -0.0 into 0.0; .4g hides floating-point noise
    # such as 6.999999999999998, which is arithmetic, not geometry.
    xi = value(hull_model.x[i]) + 0.0
    yi = value(hull_model.y[i]) + 0.0
    print(f"  rectangle {i} at ({xi:.4g}, {yi:.4g})")
print(" ", hull_size)

area_bound = sum(
    value(bigm_model.rect_length[i]) * value(bigm_model.rect_width[i])
    for i in bigm_model.rectangles
) / value(bigm_model.strip_width)

rows = [
    ("continuous variables", bigm_size["continuous"], hull_size["continuous"]),
    ("binary variables", bigm_size["binary"], hull_size["binary"]),
    ("constraints", bigm_size["constraints"], hull_size["constraints"]),
    ("LP relaxation bound at the root", root_relaxation_bound("gdp.bigm"),
     root_relaxation_bound("gdp.hull")),
    ("MILP optimum", value(bigm_model.total_length), value(hull_model.total_length)),
]

print(f"\n{'':35s} {'gdp.bigm':>10s} {'gdp.hull':>10s}")
for label, a, b in rows:
    print(f"{label:35s} {a:10.6g} {b:10.6g}")
print(f"\nArea lower bound (no packing can beat this): {area_bound:.4g}")
print("Wall-clock time is deliberately NOT reported: it is machine dependent.")


### 5-E. Visualize the packing

A table of coordinates is not a packing. Write a function that draws the solved model: one
rectangle per item, drawn at $(x_i, y_i)$ with width $L_i$ and height $H_i$, plus a line marking
the strip length $lt$. Use it on **both** solutions, one above the other.

:::{warning}
**Your figure must survive being printed in black and white.** Do not tell the rectangles apart by
colour alone --- an exam or a homework printout is greyscale, and two colours that look completely
different on screen can print as the same grey. Give each rectangle at least one *non-colour*
identity: a printed label, a hatch pattern, an edge style, or all three.
:::


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def plot_packing(model, title, ax):
    """Draw a solved strip packing.

    Greyscale safety: every rectangle carries its own index printed in the
    middle AND its own hatch pattern, so no rectangle is identified by colour
    alone. The fills are light tints so that the black label and the black
    hatch stay legible in a photocopy.
    """
    # Light tints of the house Okabe-Ito palette. They are TINTS on purpose:
    # a saturated fill swallows the black index label and the black hatch, and
    # those two are what make the figure work in a photocopy.
    fills = ["#FFFFFF", "#FFE7B3", "#CDE6F5", "#CCEBDD"]
    hatches = ["", "///", "\\\\\\", "...", "xxx", "|||", "---", "+++"]

    W = value(model.strip_width)
    lt = value(model.strip_length)

    # Add your solution here

    ax.axvline(lt, color="black", linestyle="--", linewidth=1.5)
    ax.text(lt + 0.15, W * 0.94, f"$lt$ = {lt:g}", ha="left", va="top")
    ax.axhline(0, color="black", linewidth=1.0)
    ax.axhline(W, color="black", linewidth=1.0)
    ax.set_xlim(-0.4, max(lt, 1) + 3.5)
    ax.set_ylim(-0.4, W + 0.4)
    ax.set_aspect("equal")
    ax.set_xlabel("position along the strip, $x$")
    ax.set_ylabel("across, $y$")
    ax.set_title(title)
    return ax


fig, axes = plt.subplots(2, 1, figsize=(8, 7))
plot_packing(bigm_model, "gdp.bigm", axes[0])
plot_packing(hull_model, "gdp.hull", axes[1])
fig.tight_layout()
plt.show()


### 5-F. Discussion

Two or three sentences each.

1. Both reformulations describe the *same* set of feasible packings, yet 5-E almost certainly shows
   you two **different** pictures. Explain how both can be correct.
2. How do the disjunctions affect the degree of freedom analysis? Count the binaries that
   `gdp.bigm` introduced and say where the number comes from.
3. The number of disjunctions grows as $\binom{N}{2}$ in the number of rectangles. Contrast this
   with the logic constraints in **4-E**, which did not grow at all. What is the difference between
   the two situations?
4. You wrote the model once, as a disjunction, and got two MILPs from it for free. Name one thing
   that would have gone wrong if you had typed the big-$M$ constraints by hand instead --- and
   refer to your answer in **3-B** part 3.


### Problem 5 AI use report

Use the concise report format stated at the top of the assignment.